# Lekcja 18: Zabezpieczanie agentów AI za pomocą dowodów kryptograficznych

## Ćwiczenia praktyczne w notatniku

Ten notatnik przeprowadza przez cztery ćwiczenia:

1. **Podpisz swój pierwszy dowód** dla wywołania narzędzia agenta i zweryfikuj go.
2. **Zmodyfikuj dowód** i zobacz, jak weryfikacja nie powiedzie się.
3. **Zbuduj łańcuch trzech dowodów** i potwierdź integralność łańcucha.
4. **Owiń wywołanie narzędzia Microsoft Agent Framework**, aby każda akcja emitowała dowód.

Wszystkie elementy kryptograficzne są importowane z dobrze utrzymywanych bibliotek (`pynacl` dla Ed25519, `jcs` dla canonical JSON RFC 8785, `hashlib` z biblioteki standardowej Pythona dla SHA-256). Logika dowodów jest zwykłym Pythonem, który możesz czytać i modyfikować.

Uruchamiaj bloki kodu w kolejności. Każda sekcja jest krótka i samodzielna.


## Konfiguracja

Zainstaluj dwa zależności. Obie mają licencje zezwalające na szerokie użycie (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Narzędzia pomocnicze

Te dwa narzędzia obsługują kodowanie base64url (bez dopełnienia) oraz haszowanie SHA-256 dowolnych obiektów. Pozwalają one skupić resztę notatnika na samej logice paragonu.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sekcja 1: Podpisz swój pierwszy paragon

Wyobraź sobie, że nasz agent z **Contoso Travel** właśnie wyszukał dla klienta loty z Sydney do Los Angeles. Chcemy zapisać oświadczenie bramki dotyczącą tego wywołania narzędzia, aby przyszły audytor mógł je zweryfikować pod osobno przypiętym kluczem wydawcy.

### Krok 1.1: Wygeneruj klucz podpisujący

W środowisku produkcyjnym klucz podpisujący agenta znajdowałby się w module bezpieczeństwa sprzętowego (HSM), Azure Key Vault lub podobnym chronionym magazynie. Na potrzeby tej lekcji generujemy nowy klucz w pamięci.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Krok 1.2: Utwórz dane potwierdzające

Dane zawierają wszystko, co chcemy, aby potwierdzenie poświadczało: kto działał, jakiego narzędzia użyto, z jakimi argumentami, co zostało zwrócone, na jakiej podstawie oraz kiedy. Haszujemy argumenty i wynik zamiast umieszczać je inline, aby potwierdzenie nie ujawniało poufnych treści.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Krok 1.3: Podpisz i zmontuj potwierdzenie

Trzy kroki:

1. Kanonizuj ładunek za pomocą JCS, aby dwie implementacje generujące to samo logiczne potwierdzenie produkowały identyczne bajty.
2. Podpisz kanoniczne bajty bezpośrednio przy użyciu prywatnego klucza Ed25519. PureEdDSA wewnętrznie haszuje wiadomość, więc dodatkowy pre-hash zmieniłby protokół.

Następnie podpis jest dołączany do oryginalnego ładunku, aby uzyskać ostateczne potwierdzenie.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Krok 1.4: Weryfikacja potwierdzenia

Weryfikacja odwraca proces. Usuwamy podpis, ponownie obliczamy kanoniczne bajty i sprawdzamy podpis względem klucza publicznego wystawcy.

Weryfikacja jest wykonywana offline: nie ma potrzeby wywoływania usług ani dostępu do sieci. Nadal potrzebna jest jedna rzecz oprócz potwierdzenia: klucz publiczny wystawcy, uzyskany przez kanał, któremu rewident już ufa, na przykład zestaw JWK publikowany przez wystawcę lub wartość wcześniej zablokowana. Pole `public_key` wewnątrz potwierdzenia nie może pełnić tej roli, ponieważ ten, kto tworzy potwierdzenie, również zapisuje to pole. Sekcja 2 pokazuje, jakiego rodzaju fałszerstwa to uniemożliwia.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Powinieneś zobaczyć `Receipt is valid: True`, `Pre-hashed receipt valid: False` oraz `Malformed receipts rejected: True`. Pozytywny przypadek dowodzi, że bezpośrednia ścieżka podpisywania JCS działa; negatywna kontrola sprawia, że reguła zakresu podpisu jest wykonalna, zamiast pozostawać w formie prozy; ostatnia linia pokazuje błędne dane wejściowe zwracające `False` zamiast zgłaszania wyjątku.


## Sekcja 2: Manipulacja paragonem

Cały sens paragonów polega na tym, że są odporne na manipulacje. Udowodnijmy to.

Zmienimy dokładnie jeden znak na paragonie i zobaczymy, jak weryfikacja się nie powiedzie.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Co się właśnie stało?

Kiedy zmieniliśmy `policy_id`, zmieniły się kanoniczne bajty. Podpis (który obejmował oryginalne kanoniczne bajty) już nie pasuje. Weryfikacja poprawnie zwraca `False`.

Nie ma możliwości zmiany podpisanego pola, czyli czegokolwiek poza obiektem `signature`, i jednoczesnego zweryfikowania paragonu za pomocą klucza publicznego wystawcy, chyba że atakujący posiada prywatny klucz wystawcy. Sam obiekt `signature` nie jest podpisany: `sig` to podpis, a `verify_receipt` ignoruje `public_key`, więc samo edytowanie tego pola nic nie zmienia. Dopóki prywatny klucz jest w sejfie kluczy, a weryfikatorzy korzystają z niezależnie opublikowanego klucza wystawcy, zmiana podpisanego pola jest niemożliwa do ukrycia. Ten warunek ma znaczenie: w następnej komórce pokazano zmodyfikowany paragon, który weryfikuje się idealnie przeciwko innemu kluczowi.

Spróbuj sam: zmodyfikuj `tool_name` lub `agent_id` albo `timestamp` w powyższej komórce i uruchom ponownie. Każda zmiana podpisanego pola generuje nieważny paragon.


### Fałszerstwo, którego podpis nie wykrywa

Powyższy atakujący nie podpisał na nowo. Bardziej ostrożny by to zrobił: zmieniłby paragon,
podpisał go kluczem wygenerowanym przez siebie i umieścił ten klucz w polu
`public_key`. W efekcie powstaje paragon, którego podpis jest całkowicie autentyczny
dla jego zawartości.

Uruchom to i zobacz, który z dwóch mechanizmów to wykryje.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Powinieneś zobaczyć `True`, a następnie `False`. Trzy linie `assert` tworzą
sprawdzenie: komórka zgłasza błąd, jeśli podróbka kiedykolwiek zostanie zweryfikowana w oparciu o
klucz wydawcy.

Podróbka jest wewnętrznie spójna. Jej podpis naprawdę zgadza się z jej
zawartością oraz posiadanym kluczem, więc weryfikator, który odczytuje klucz z
pokwitowania, zgłasza go jako ważne. Nic w pokwitowaniu nie wygląda na niewłaściwe,
ponieważ nic w nim nie jest niespójne.

Jedyną rzeczą, która ją wykrywa, jest sprawdzenie względem klucza, na który atakujący nie miał
możliwości wpływu. Dlatego `verify_receipt` przyjmuje klucz jako argument: funkcja o sygnaturze
`verify_receipt(receipt)` nie może być napisana bezpiecznie,
ponieważ jedyny klucz, do którego mogłaby dotrzeć, to ten dostarczony przez atakującego.

To samo pokwitowanie jest zapisywane jako `sample_receipts/04_resigned_receipt.json`, jeśli chcesz
obejrzeć je poza notebookiem.


## Sekcja 3: Łączenie potwierdzeń w łańcuch

Pojedyncze potwierdzenie chroni jedną akcję. Większość agentów wykonuje wiele akcji. Aby cały ciąg był odporny na manipulacje, łączymy każde potwierdzenie z poprzednim, dołączając skrót poprzedniego potwierdzenia do danych nowego potwierdzenia.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jeśli ktoś usunie lub zmieni kolejność potwierdzeń w przedstawionym ciągu, kolejny łącznik zostaje zerwany. Wykrycie ucięcia lub zmiany sufiksu kontrolowanego przez podpisującego wymaga również przypiętej obecnej głowy, zewnętrznego punktu kontrolnego lub uwierzytelnionego zobowiązania populacji.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Teraz przerwij łańcuch, manipulując środkowym paragonem i ponownie zweryfikuj. Zmieniony paragon nie przechodzi kontroli podpisu, A następny paragon nie przechodzi kontroli łącza łańcucha (ponieważ jego `previous_receipt_hash` nie odpowiada już zmienionemu skrótowi środkowego paragonu).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Potwierdzenie 0 nadal się weryfikuje (nie zostało zmodyfikowane i nie ma poprzednika, od którego miałoby zależeć). Potwierdzenie 1 nie przechodzi sprawdzenia podpisu, ponieważ zmieniliśmy `tool_args_hash`. Potwierdzenie 2 nie przechodzi sprawdzenia łańcucha, ponieważ jego `previous_receipt_hash` został obliczony względem oryginalnego (już zmodyfikowanego) potwierdzenia 1.

Nawet jeśli atakujący ponownie podpisze zmodyfikowane potwierdzenie 1, nowy podpis nie przejdzie sprawdzenia przeciwko kluczowi wystawcy, chyba że posiada prywatny klucz wystawcy (ponowne podpisanie własnym kluczem nie działa, jak pokazano w Sekcji 2), a brak dopasowania łańcucha w potwierdzeniu 2 nadal ujawni manipulację. Aby ukryć zmianę, atakujący musiałby ponownie podpisać każde potwierdzenie od punktu modyfikacji wzwyż, co wymaga posiadania prywatnego klucza wystawcy.


## Sekcja 4: Otocz wywołanie narzędzia agenta podpisywaniem potwierdzenia

W rzeczywistym wdrożeniu nie chcesz, aby każdy twórca agenta pamiętał o wywołaniu `make_receipt`. Chcesz, aby podpisywanie potwierdzenia było automatyczne przy każdym wywołaniu narzędzia.

Oto najprostszy wzorzec: klasa opakowująca, która przyjmuje dowolną wywoływalną funkcję narzędzia i zwraca jej wersję emitującą potwierdzenia. Ten wzorzec dostosowuje się do dowolnego frameworka agenta, w tym Microsoft Agent Framework (`agent_framework.foundry`).

Jeśli nie masz skonfigurowanego projektu Microsoft Foundry, poniższy lokalny mock nadal demonstruje ten wzorzec.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integracja z Microsoft Agent Framework

Powyższy wrapper `ReceiptedTool` jest niezależny od frameworka. Aby użyć go wewnątrz agenta zbudowanego za pomocą Microsoft Agent Framework, zarejestruj opakowaną funkcję jako narzędzie. Szkic (zastąpiłbyś mock rzeczywistą rejestracją narzędzia Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Framework agenta nie musi nic wiedzieć o potwierdzeniach. Podpisywanie potwierdzeń jest opakowane wokół narzędzia, a nie wbudowane w framework. W ten sposób dodajesz pochodzenie do istniejącego kodu agenta bez przepisywania agenta.


## Recap and stretch challenge

You have:

- Generated an Ed25519 key pair.
- Built and signed a receipt for an agent tool call.
- Verified the receipt offline using only the issuer's public key, obtained independently of the receipt.
- Tampered with a receipt and observed verification fail.
- Built a hash-chained sequence of three receipts.
- Tampered with the middle of the chain and observed both signature failure and chain-link failure.
- Wrapped a tool function with automatic receipt signing.

**Stretch challenge.** Extend the receipt schema with a `request_id` field (a UUID for distributed tracing). Update `make_receipt` to include it, and confirm receipts still verify end to end. Then modify the field after signing and confirm verification fails. This forces you to internalize how every byte of the canonical encoding contributes to the signature.

**Important boundary.** Under a pinned key and chain context, receipts prove signature attribution, content integrity, and a link to the presented predecessor. They do NOT prove execution, source truth, trusted time, completeness, policy evaluation, or correctness. Receipts are a foundation. Governance is the system you build on top.

Read the lesson README again with that boundary in mind. The most common mistake teams make with receipts is assuming "we have receipts" means "we are governed." It does not. Receipts make agent behavior auditable. They do not make it correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
